In [ ]:
%run ./NB_Config_SELLING_SYSTEM

from datetime import datetime
import uuid
from pyspark.sql import functions as F

PIPELINE_RUN_ID = str(uuid.uuid4())
start_time = datetime.utcnow()
results = []
skipped = []

def ingest_delta(source_name, target_name=None, columns=None):
    target_name = target_name or source_name
    source_path = f'{SOURCE_LH_ABFSS}/{SOURCE_SCHEMA}/{source_name}'
    bronze_path = f'{BRONZE_LH_ABFSS}/{BRONZE_SCHEMA}/{target_name}'
    dataframe = spark.read.format('delta').load(source_path)
    if columns:
        available = [column_name for column_name in columns if column_name in dataframe.columns]
        if available:
            dataframe = dataframe.select(*available)
    row_count = dataframe.count()
    (dataframe
        .withColumn('_PIPELINE_NAME', F.lit(PIPELINE_NAME))
        .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
        .withColumn('_INGESTED_AT', F.current_timestamp())
        .withColumn('_IS_DELETED', F.lit(False))
        .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(bronze_path))
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {BRONZE_SCHEMA}.{target_name} "
        f"USING DELTA LOCATION '{bronze_path}'"
    )
    return row_count

for config_name, table_config in SELLING_TABLE_CONFIGS.items():
    source_name = table_config['source_table']
    try:
        row_count = ingest_delta(source_name, columns=table_config['silver_cols'])
        results.append({'table': config_name, 'source': source_name, 'rows': row_count})
        print(f'{config_name}: {source_name} loaded {row_count:,} rows')
    except Exception as exc:
        skipped.append({'table': config_name, 'source': source_name, 'error': str(exc)})
        print(f'{config_name}: skipped ({exc})')

# External inputs used by the Alteryx workflows are optional Bronze tables.
for reference_table, reference_columns in WORKFLOW_REFERENCE_TABLES.items():
    try:
        row_count = ingest_delta(reference_table, columns=reference_columns)
        results.append({'table': reference_table, 'rows': row_count})
        print(f'{reference_table}: reference loaded {row_count:,} rows')
    except Exception as exc:
        skipped.append({'table': reference_table, 'error': str(exc)})
        print(f'{reference_table}: reference skipped ({exc})')

print(f'Bronze complete: {len(results)} loaded, {len(skipped)} skipped')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
duration = (datetime.utcnow() - start_time).total_seconds()
print('\n' + '=' * 70)
print('SELLING SYSTEM BRONZE — SUMMARY')
print('=' * 70)
print(f'{"Table":<35} {"Rows":>12}')
print('-' * 70)
for result in results:
    print(f'{result["table"]:<35} {result["rows"]:>12,}')
print('-' * 70)
print(f'Loaded  : {len(results)}')
print(f'Skipped : {len(skipped)}')
print(f'Run ID  : {PIPELINE_RUN_ID}')
print(f'Duration: {duration:.1f}s')